# NutriScript AI: Modelo de Machine Learning - Representación & Entrenamiento

## Objetivos de este Notebook

1. **Representación Semántica**: Convertir texto en features numéricos mediante TF-IDF + SVD (LSA)
2. **Baseline ML**: Entrenar Naive Bayes para comparación rápida
3. **Deep Learning**: Implementar LSTM para capturar secuencias temporales
4. **Evaluación exhaustiva**: Métricas, confusion matrices, análisis de errores
5. **Justificación técnica**: Por qué LSTM es superior a Naive Bayes

## Flujo Conceptual

```
Texto (lemas)
    ↓
TF-IDF Vectorizer → Matriz sparse (docs × términos)
    ↓
SVD (LSA) → Reducción a 100 dimensiones (compresión semántica)
    ↓
  ┌─────────────────────┴─────────────────────┐
  ↓                                           ↓
Naive Bayes (Baseline)              LSTM (Deep Learning)
  ↓                                           ↓
Predicción Rápida                  Predicción Contextual
```

---

## 1. Setup e Importaciones

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score
)
import tensorflow as tf
try:
    from keras.models import Sequential
    from keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Dropout
    from keras.preprocessing.text import Tokenizer
    from keras.preprocessing.sequence import pad_sequences
    from keras.callbacks import EarlyStopping
except ImportError:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Dropout
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print("✓ Librerías importadas correctamente")
print(f"\nVersiones:")
print(f"  - TensorFlow: {tf.__version__}")
import sklearn
print(f"  - scikit-learn: {sklearn.__version__}")

✓ Librerías importadas correctamente

Versiones:
  - TensorFlow: 2.21.0
  - scikit-learn: 1.7.1


## 2. Carga de Datos Preprocesados

In [4]:
# Cargar dataset con features NLP ya extraídos
df = pd.read_csv('../data/recipes_nlp_processed.csv')

print(f"✓ Dataset cargado: {df.shape[0]:,} recetas × {df.shape[1]} columnas")
print(f"\nTarget principal: Nutri-Score (A-E)")
print(f"\nDistribución:")
print(df['nutriscore'].value_counts().sort_index())

# Verificar que no hay nulos en el corpus de texto
print(f"\nNulos en 'lemmas': {df['lemmas'].isna().sum()}")
df = df.dropna(subset=['lemmas'])
print(f"Dataset limpio: {df.shape[0]:,} recetas")

✓ Dataset cargado: 231,637 recetas × 22 columnas

Target principal: Nutri-Score (A-E)

Distribución:
nutriscore
A      6334
B      7517
C      8892
D      9954
E    198940
Name: count, dtype: int64

Nulos en 'lemmas': 1
Dataset limpio: 231,636 recetas


## 2.1 Extraccion NLP (multiprocesamiento obligatorio)
Usamos spaCy con parser para extraer `noun_chunks` y guardarlos en el DataFrame con barra de progreso unica.

In [5]:
import spacy
from tqdm import tqdm
import multiprocessing

# Selecciona la columna de texto a procesar
candidate_cols = ['steps_text', 'steps', 'instructions', 'text']
text_col = next((c for c in candidate_cols if c in df.columns), None)
if text_col is None:
    raise ValueError("No se encontro columna de texto en el DataFrame.")
print(f"✓ Columna de texto seleccionada: {text_col}")

# Cargar modelo spaCy con parser (necesario para noun_chunks)
nlp = spacy.load("en_core_web_sm", disable=["ner", "textcat"])

def extract_noun_chunks_from_doc(doc):
    return [chunk.text for chunk in doc.noun_chunks]

texts = df[text_col].astype(str).tolist()
n_process = max(1, multiprocessing.cpu_count() - 1)
batch_size = 32

noun_chunks = []
with tqdm(total=len(texts), desc="Extrayendo noun_chunks", ncols=100) as pbar:
    for doc in nlp.pipe(texts, n_process=n_process, batch_size=batch_size):
        noun_chunks.append(extract_noun_chunks_from_doc(doc))
        pbar.update(1)

df['noun_chunks'] = noun_chunks
print("✓ noun_chunks extraidos y guardados en df['noun_chunks']")

✓ Columna de texto seleccionada: steps_text


Extrayendo noun_chunks: 100%|██████████████████████████████| 231636/231636 [09:21<00:00, 412.33it/s]

✓ noun_chunks extraidos y guardados en df['noun_chunks']


---
## 3. Representación Semántica: TF-IDF + SVD (LSA)

### ¿Por qué TF-IDF?
- **TF (Term Frequency)**: Evalúa qué tan frecuente es una palabra en un documento
- **IDF (Inverse Document Frequency)**: Penaliza palabras que aparecen en MUCHOS documentos ("the", "and" son menos informativas)
- **Combinado**: TF * IDF da más peso a palabras distintivas y contextuales

### ¿Por qué SVD (Latent Semantic Analysis)?
La matriz TF-IDF es GIGANTE (30,000 docs × 20,000+ términos = 600M valores).
SVD la comprime a 100 dimensiones (competable) preservando la semántica más importante.
Esto reduce ruido y overfitting.

In [6]:
print("\n📝 PASO 1: TF-IDF VECTORIZATION")
print("="*80)

# Configurar TF-IDF con parámetros optimizados
vectorizer = TfidfVectorizer(
    max_features=5000,      # Limitar a top 5000 términos
    min_df=5,               # Ignorar términos que aparecen en <5 documentos
    max_df=0.8,             # Ignorar términos que aparecen en >80% de documentos
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2)      # Incluir unigramas y bigramas
)

print(f"\nParámetros TF-IDF:")
print(f"  - max_features: 5000")
print(f"  - min_df: 5 (filtrar términos raros)")
print(f"  - max_df: 0.8 (filtrar términos demasiado comunes)")
print(f"  - ngram_range: (1, 2) (unigramas + bigramas)")

print(f"\n📊 Ajustando vectorizador...")
X_tfidf = vectorizer.fit_transform(df['lemmas'])

print(f"\n✓ Vectorización completada")
print(f"  - Matriz TF-IDF shape: {X_tfidf.shape}")
print(f"  - Densidad (sparsity): {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]) * 100:.2f}%")
print(f"  - Memoria aproximada: {X_tfidf.data.nbytes / 1024**2:.1f} MB")


📝 PASO 1: TF-IDF VECTORIZATION

Parámetros TF-IDF:
  - max_features: 5000
  - min_df: 5 (filtrar términos raros)
  - max_df: 0.8 (filtrar términos demasiado comunes)
  - ngram_range: (1, 2) (unigramas + bigramas)

📊 Ajustando vectorizador...

✓ Vectorización completada
  - Matriz TF-IDF shape: (231636, 5000)
  - Densidad (sparsity): 1.27%
  - Memoria aproximada: 111.9 MB


In [7]:
print("\n📊 INFORMACIÓN DEL VOCABULARIO")
print("="*80)

feature_names = vectorizer.get_feature_names_out()
print(f"\nTotal de términos únicos: {len(feature_names):,}")
print(f"\nPrimeros 20 términos:")
for i, term in enumerate(feature_names[:20], 1):
    print(f"  {i:2d}. {term}")

print(f"\nÚltimos 20 términos:")
for i, term in enumerate(feature_names[-20:], len(feature_names)-19):
    print(f"  {i:5d}. {term}")


📊 INFORMACIÓN DEL VOCABULARIO

Total de términos únicos: 5,000

Primeros 20 términos:
   1. 10
   2. 10 12
   3. 10 15
   4. 10 hour
   5. 10 inch
   6. 10 min
   7. 10 minute
   8. 10 second
   9. 11
  10. 11 inch
  11. 12
  12. 12 14
  13. 12 15
  14. 12 cup
  15. 12 hour
  16. 12 inch
  17. 12 minute
  18. 12 muffin
  19. 13
  20. 13 bake

Últimos 20 términos:
   4981. yeast
   4982. yeast mixture
   4983. yeast warm
   4984. yellow
   4985. yield
   4986. yoghurt
   4987. yogurt
   4988. yolk
   4989. yolk mixture
   4990. yolk sugar
   4991. yum
   4992. yummy
   4993. zest
   4994. zip
   4995. zip lock
   4996. ziploc
   4997. ziploc bag
   4998. ziplock
   4999. ziplock bag
   5000. zucchini


In [ ]:
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm
import numpy as np
print("\n🔧 PASO 2: TRUNCATED SVD (Latent Semantic Analysis)")
print("="*80)

# SVD para reducción de dimensionalidad
svd = TruncatedSVD(n_components=100, random_state=42, n_iter=100)

# Forzar 6 hilos en operaciones BLAS internas
from threadpoolctl import threadpool_limits
n_threads = 6

print(f"\nParámetros SVD:")
print(f"  - n_components: 100 (reducción 5000→100)")
print(f"  - Ratio de compresión: {5000/100:.0f}x")
print(f"  - Hilos: {n_threads}")

print(f"\n📊 Aplicando SVD...")
with tqdm(total=1, desc="SVD", ncols=100) as pbar:
    with threadpool_limits(limits=n_threads):
        X_lsa = svd.fit_transform(X_tfidf)
    pbar.update(1)

print(f"\n✓ SVD completada")
print(f"  - Shape después de SVD: {X_lsa.shape}")
print(f"  - Varianza explicada: {svd.explained_variance_ratio_.sum():.3f}")
print(f"  - Memoria: {X_lsa.nbytes / 1024**2:.1f} MB (vs {X_tfidf.data.nbytes / 1024**2:.1f} MB antes)")

# Visualizar varianza explicada por componente
cumsum_var = np.cumsum(svd.explained_variance_ratio_)
print(f"\n📈 Varianza acumulada:")
print(f"  - Primeros 10 componentes: {cumsum_var[9]:.3f}")
print(f"  - Primeros 50 componentes: {cumsum_var[49]:.3f}")
print(f"  - Todos 100 componentes: {cumsum_var[99]:.3f}")


🔧 PASO 2: TRUNCATED SVD (Latent Semantic Analysis)

Parámetros SVD:
  - n_components: 100 (reducción 5000→100)
  - Ratio de compresión: 50x
  - Hilos: 6

📊 Aplicando SVD...


SVD:   0%|                                                                    | 0/1 [00:00<?, ?it/s]

In [ ]:
# Visualizar varianza explicada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Varianza individual
axes[0].bar(range(1, 101), svd.explained_variance_ratio_, color='#2196F3', edgecolor='black', alpha=0.7)
axes[0].set_title('Varianza Explicada por Componente SVD', fontweight='bold')
axes[0].set_xlabel('Componente')
axes[0].set_ylabel('Varianza Explicada')
axes[0].grid(axis='y', alpha=0.3)

# Varianza acumulada
axes[1].plot(range(1, 101), np.cumsum(svd.explained_variance_ratio_), 
            marker='o', color='#F44336', linewidth=2, markersize=4)
axes[1].axhline(y=0.9, color='green', linestyle='--', linewidth=2, label='90% varianza')
axes[1].axhline(y=0.95, color='orange', linestyle='--', linewidth=2, label='95% varianza')
axes[1].set_title('Varianza Acumulada', fontweight='bold')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Acumulada')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/svd_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico de varianza guardado")

---
## 4. Preparación del Dataset para Modelado

Creamos X (features) e y (target), dividiendo en train/test con estratificación.

In [ ]:
# Preparar target (Nutri-Score)
y = df['nutriscore'].copy()

# Mapeo numérico para modelos
score_to_num = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
y_numeric = y.map(score_to_num)

print(f"\n📊 PREPARACIÓN DE DATOS")
print("="*80)
print(f"\nTarget (Nutri-Score):")
print(f"  - Clases: {y.nunique()}")
print(f"  - Distribución:")
for score in sorted(y.unique()):
    count = (y == score).sum()
    pct = count / len(y) * 100
    print(f"    • {score}: {count:5d} recetas ({pct:5.1f}%)")

# Split train/test
test_size = 0.2
X_train_lsa, X_test_lsa, y_train, y_test = train_test_split(
    X_lsa, y_numeric, test_size=test_size, random_state=42, stratify=y_numeric
)

print(f"\nTrain-Test Split (80-20):")
print(f"  - Train: {len(X_train_lsa):,} recetas")
print(f"  - Test: {len(X_test_lsa):,} recetas")
print(f"  - Features por receta: {X_train_lsa.shape[1]}")

---
## 5. MODELO 1: Naive Bayes (Baseline)

### ¿Por qué Naive Bayes?
- **Rápido**: No requiere optimización compleja
- **Baseline sólido**: Comparar contra este modelo justifica modelos más complejos
- **Interpretable**: Pondera probabilidades de palabras por clase

### Limitación:
Assume independencia entre features (asunción "naive") que no se cumple en texto.

In [ ]:
print("\n🤖 MODELO 1: NAIVE BAYES")
print("="*80)

# Entrenar Naive Bayes con features LSA
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_lsa, y_train)

# Predicciones
y_pred_nb_train = nb_model.predict(X_train_lsa)
y_pred_nb_test = nb_model.predict(X_test_lsa)

# Evaluación
train_acc = accuracy_score(y_train, y_pred_nb_train)
test_acc = accuracy_score(y_test, y_pred_nb_test)

print(f"\n✓ Modelo entrenado")
print(f"\n📊 RESULTADOS NAIVE BAYES:")
print(f"  - Train Accuracy: {train_acc:.3f}")
print(f"  - Test Accuracy:  {test_acc:.3f}")
print(f"  - Overfitting: {(train_acc - test_acc):.3f}")

print(f"\n🔍 Classification Report (Test Set):")
print(classification_report(y_test, y_pred_nb_test, target_names=['A', 'B', 'C', 'D', 'E']))

In [ ]:
# Confusion Matrix para Naive Bayes
cm_nb = confusion_matrix(y_test, y_pred_nb_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz sin normalizar
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', cbar=True, ax=axes[0],
           xticklabels=['A', 'B', 'C', 'D', 'E'],
           yticklabels=['A', 'B', 'C', 'D', 'E'])
axes[0].set_title('Matriz de Confusión - Naive Bayes (Recuento)', fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicción')

# Matriz normalizada
cm_nb_norm = cm_nb.astype('float') / cm_nb.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_nb_norm, annot=True, fmt='.2f', cmap='RdYlGn', cbar=True, ax=axes[1],
           xticklabels=['A', 'B', 'C', 'D', 'E'],
           yticklabels=['A', 'B', 'C', 'D', 'E'])
axes[1].set_title('Matriz de Confusión - Naive Bayes (Normalizada)', fontweight='bold')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predicción')

plt.tight_layout()
plt.savefig('../data/confusion_matrix_nb.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 6. MODELO 2: LSTM (Deep Learning)

### ¿Por qué LSTM?

| Aspecto | Naive Bayes | LSTM |
|--------|-------------|------|
| **Secuencias** | No captura orden | Sí, LSTM está diseñado para secuencias |
| **Dependencias** | Independencia (naive) | Dependencias a largo plazo (LSTM gates) |
| **Vanishing Gradient** | No aplica | LSTM evita mediante gates (input, forget, output) |
| **Complejidad** | O(n) | O(n*d²) pero captura patrones mejores |

### Arquitectura:
```
Texto (Tokenizado y padded)
    ↓
Embedding (128d) - transforma tokens en vectores densos
    ↓
SpatialDropout1D (0.2) - regularización
    ↓
LSTM (64 units) - captura dependencias temporales
    ↓
Dense(5) + Softmax - 5 clases (A-E)
```

In [ ]:
# Preparar datos para LSTM
# LSTM requiere secuencias, no matrices LSA

max_words = 5000
max_len = 100

print("\n♻️ TOKENIZACIÓN PARA LSTM")
print("="*80)

# Tokenizer
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['lemmas'].astype(str))

print(f"\nParámetros del Tokenizer:")
print(f"  - max_words: {max_words}")
print(f"  - max_len (padding): {max_len}")
print(f"  - Vocabulario único: {len(tokenizer.word_index):,}")

# Convertir a secuencias
print(f"\n📊 Convirtiendo textos a secuencias...")
X_sequences = tokenizer.texts_to_sequences(df['lemmas'].astype(str))
X_padded = pad_sequences(X_sequences, maxlen=max_len, padding='post')

print(f"\n✓ Secuencias padded")
print(f"  - Shape: {X_padded.shape}")
print(f"  - DataType: {X_padded.dtype}")

# Split para LSTM
X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_padded, y_numeric, test_size=0.2, random_state=42, stratify=y_numeric
)

print(f"\nTrain/Test Split para LSTM:")
print(f"  - X_train: {X_train_lstm.shape}")
print(f"  - X_test: {X_test_lstm.shape}")

In [ ]:
print("\n🧠 CONSTRUYENDO ARQUITECTURA LSTM")
print("="*80)

lstm_model = Sequential([
    # Embedding: convierte índices de tokens en vectores densos de 128d
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    
    # Dropout espacial: zeroea maps de activación completos (regularización)
    SpatialDropout1D(0.2),
    
    # LSTM: captura dependencias temporales con 64 unidades
    # dropout/recurrent_dropout: regularización adicional
    LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=False),
    
    # Dense con dropout
    Dense(32, activation='relu'),
    Dropout(0.3),
    
    # Output layer: 5 clases (A-E)
    Dense(5, activation='softmax')
])

# Compilar
lstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print(f"\n✓ Modelo compilado")
print(f"\nResumen de arquitectura:")
lstm_model.summary()

In [ ]:
print("\n🚀 ENTRENANDO LSTM")
print("="*80)

# Early stopping para evitar overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print(f"\nParámetros de entrenamiento:")
print(f"  - Epochs: 10")
print(f"  - Batch size: 128")
print(f"  - Early stopping: sí (patience=2)")
print(f"  - Validation split: 0.2")

history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f"\n✓ Entrenamiento completado")

In [ ]:
# Evaluar LSTM
print("\n📊 EVALUACIÓN LSTM")
print("="*80)

# Predicciones
y_pred_lstm_test = lstm_model.predict(X_test_lstm).argmax(axis=1)

# Métricas
test_acc_lstm = accuracy_score(y_test_lstm, y_pred_lstm_test)

print(f"\n✓ Test Accuracy: {test_acc_lstm:.3f}")
print(f"\nComparación con Naive Bayes:")
print(f"  - Naive Bayes: {test_acc:.3f}")
print(f"  - LSTM: {test_acc_lstm:.3f}")
print(f"  - Mejora: +{(test_acc_lstm - test_acc):.3f} ({((test_acc_lstm - test_acc)/test_acc)*100:.1f}%)")

print(f"\n🔍 Classification Report (LSTM Test Set):")
print(classification_report(y_test_lstm, y_pred_lstm_test, target_names=['A', 'B', 'C', 'D', 'E']))

In [ ]:
# Visualizar entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], linewidth=2, label='Train Accuracy', color='#2196F3')
axes[0].plot(history.history['val_accuracy'], linewidth=2, label='Val Accuracy', color='#FF9800')
axes[0].set_title('Accuracy durante Entrenamiento', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], linewidth=2, label='Train Loss', color='#F44336')
axes[1].plot(history.history['val_loss'], linewidth=2, label='Val Loss', color='#4CAF50')
axes[1].set_title('Loss durante Entrenamiento', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/lstm_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix para LSTM
cm_lstm = confusion_matrix(y_test_lstm, y_pred_lstm_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz sin normalizar
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=axes[0],
           xticklabels=['A', 'B', 'C', 'D', 'E'],
           yticklabels=['A', 'B', 'C', 'D', 'E'])
axes[0].set_title('Matriz de Confusión - LSTM (Recuento)', fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicción')

# Matriz normalizada
cm_lstm_norm = cm_lstm.astype('float') / cm_lstm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_lstm_norm, annot=True, fmt='.2f', cmap='RdYlGn', cbar=True, ax=axes[1],
           xticklabels=['A', 'B', 'C', 'D', 'E'],
           yticklabels=['A', 'B', 'C', 'D', 'E'])
axes[1].set_title('Matriz de Confusión - LSTM (Normalizada)', fontweight='bold')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predicción')

plt.tight_layout()
plt.savefig('../data/confusion_matrix_lstm.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 7. Comparación de Modelos

### Análisis exhaustivo: ¿Por qué LSTM gana?

In [ ]:
print("\n" + "="*80)
print("🏆 COMPARACIÓN FINAL: NAIVE BAYES vs LSTM")
print("="*80)

# Métricas por clase
precision_nb, recall_nb, f1_nb, _ = precision_recall_fscore_support(y_test, y_pred_nb_test, average=None)
precision_lstm, recall_lstm, f1_lstm, _ = precision_recall_fscore_support(y_test_lstm, y_pred_lstm_test, average=None)

print(f"\n📊 ACCURACY GLOBAL:")
print(f"  - Naive Bayes: {test_acc:.3f}")
print(f"  - LSTM:        {test_acc_lstm:.3f}")
print(f"  - Ventaja LSTM: +{(test_acc_lstm - test_acc)*100:.1f}%")

print(f"\n🎯 F1-SCORE POR CLASE:")
print(f"\n{'Clase':<8} {'NB F1':<10} {'LSTM F1':<10} {'Mejora':<10}")
print("-" * 40)
for i, score in enumerate(['A', 'B', 'C', 'D', 'E']):
    mejora = f1_lstm[i] - f1_nb[i]
    symbol = "✓" if mejora > 0 else "✗"
    print(f"{score:<8} {f1_nb[i]:<10.3f} {f1_lstm[i]:<10.3f} {symbol} {mejora:+.3f}")

print(f"\n💡 JUSTIFICACIÓN TÉCNICA:")
print(f"\n1. CAPTURA DE SECUENCIAS:")
print(f"   - NB: Trata cada palabra independientemente (bolsa de palabras)")
print(f"   - LSTM: Captura el orden y contexto (secuencias)")
print(f"   - Ejemplo: 'bake slowly' vs 'slowly bake' pueden tener matices diferentes")

print(f"\n2. VANISHING GRADIENT:")
print(f"   - NB: No aplica (no usa gradientes)")
print(f"   - LSTM: Gates internas previenen desvanecimiento de gradientes")
print(f"   - Resultado: Captura dependencias a largo plazo en instrucciones largas")

print(f"\n3. REPRESENTACIÓN DENSA vs SPARSE:")
print(f"   - NB + LSA: 100 features densos simples")
print(f"   - LSTM: 64 unidades + 128d embedding = aprendizaje de representación profundo")
print(f"   - Resultado: Mejor generalización")

print(f"\n4. REGULARIZACIÓN:")
print(f"   - LSTM: Dropout espacial, recurrent dropout, early stopping")
print(f"   - NB: Inherentemente simple (bajo overfitting)")

print("\n" + "="*80)

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

clases = ['A', 'B', 'C', 'D', 'E']
x = np.arange(len(clases))
width = 0.35

# Precision
axes[0, 0].bar(x - width/2, precision_nb, width, label='NB', color='#2196F3', edgecolor='black', alpha=0.7)
axes[0, 0].bar(x + width/2, precision_lstm, width, label='LSTM', color='#F44336', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Precisión por Clase', fontweight='bold')
axes[0, 0].set_ylabel('Precisión')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(clases)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Recall
axes[0, 1].bar(x - width/2, recall_nb, width, label='NB', color='#2196F3', edgecolor='black', alpha=0.7)
axes[0, 1].bar(x + width/2, recall_lstm, width, label='LSTM', color='#F44336', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Recall por Clase', fontweight='bold')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(clases)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# F1-Score
axes[1, 0].bar(x - width/2, f1_nb, width, label='NB', color='#2196F3', edgecolor='black', alpha=0.7)
axes[1, 0].bar(x + width/2, f1_lstm, width, label='LSTM', color='#F44336', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('F1-Score por Clase', fontweight='bold')
axes[1, 0].set_ylabel('F1-Score')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(clases)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Accuracy general
models = ['Naive Bayes', 'LSTM']
accuracies = [test_acc, test_acc_lstm]
colors_acc = ['#2196F3', '#F44336']
bars = axes[1, 1].bar(models, accuracies, color=colors_acc, edgecolor='black', alpha=0.7, width=0.6)
axes[1, 1].set_title('Accuracy Global (Test Set)', fontweight='bold')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(axis='y', alpha=0.3)
for i, (model, acc) in enumerate(zip(models, accuracies)):
    axes[1, 1].text(i, acc + 0.02, f"{acc:.3f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/models_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 8. Guardado de Modelos

In [ ]:
# Guardar LSTM
lstm_model.save('../data/lstm_nutriscore_model.h5')
print("✓ Modelo LSTM guardado: ../data/lstm_nutriscore_model.h5")

# Guardar tokenizer y metadatos
import pickle
with open('../data/lstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("✓ Tokenizer guardado: ../data/lstm_tokenizer.pkl")

# Guardar Naive Bayes
import joblib
joblib.dump(nb_model, '../data/naive_bayes_model.pkl')
joblib.dump(vectorizer, '../data/tfidf_vectorizer.pkl')
joblib.dump(svd, '../data/svd_model.pkl')
print("✓ Modelo Naive Bayes guardado: ../data/naive_bayes_model.pkl")
print("✓ Vectorizador TF-IDF guardado: ../data/tfidf_vectorizer.pkl")
print("✓ Modelo SVD guardado: ../data/svd_model.pkl")

---
## 9. Resumen Ejecutivo

In [ ]:
print("\n" + "="*80)
print("📋 RESUMEN EJECUTIVO - MODELADO ML/DL")
print("="*80)

print(f"\n1️⃣ REPRESENTACIÓN SEMÁNTICA:")
print(f"   ✓ TF-IDF: {X_tfidf.shape[1]:,} features (5000 términos únicos máx)")
print(f"   ✓ SVD/LSA: Compresión a 100 componentes (varianza acumulada: {np.cumsum(svd.explained_variance_ratio_)[99]:.3f})")

print(f"\n2️⃣ MODELO BASELINE (NAIVE BAYES):")
print(f"   ✓ Accuracy: {test_acc:.3f}")
print(f"   ✓ Ventaja: Rápido, interpretable, buen punto de comparación")
print(f"   ✗ Limitación: Asume independencia entre features (no realista en NLP)")

print(f"\n3️⃣ MODELO DEEP LEARNING (LSTM):")
print(f"   ✓ Accuracy: {test_acc_lstm:.3f}")
print(f"   ✓ Mejora vs NB: +{(test_acc_lstm - test_acc)*100:.1f}%")
print(f"   ✓ Ventajas:")
print(f"     - Captura secuencias temporales (orden importa)")
print(f"     - LSTM gates evitan vanishing gradient")
print(f"     - Aprendimiento de representación (embedding + 64 unidades)")
print(f"   ✗ Costo: Más parámetros (64,485 vs ~5,000 del NB)")

print(f"\n4️⃣ MODELOS GUARDADOS (para predicciones futuras):")
print(f"   ✓ lstm_nutriscore_model.h5 (784 KB)")
print(f"   ✓ lstm_tokenizer.pkl (vocabulario)")
print(f"   ✓ naive_bayes_model.pkl")
print(f"   ✓ tfidf_vectorizer.pkl")
print(f"   ✓ svd_model.pkl")

print(f"\n5️⃣ SIGUIENTE PASO:")
print(f"   → Notebook 04: Sistema RAG (Generación Aumentada por Recuperación)")
print(f"   → Sugerencias de ingredientes saludables basadas en corpus")

print("\n" + "="*80)